# Test Results — Performance by Training Coverage

Main analysis runs on the **well-trained** `(station, variable)` pairs (present in ≥ 50 %
of the training years). The pairs that are **missing in training** are analysed
separately **at the end**.

Errors are un-normalised to **physical units** using the exact per-`(station, variable)`
std the pipeline applied (per-station, with a global fallback for pairs with < 50
training observations), and expressed alongside each variable's global spread so the
**scale of the error is interpretable**.

Sections:
1. General metrics per variable (well-trained), physical + scale reference
2. Outlier stations/variables (unusually large errors)
3. Average error by forecast timestep and variable
4. *(end)* Metrics on the missing-in-training pairs

In [ ]:
# ── Config & imports ─────────────────────────────────────────────────────────
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ── Project bootstrap ────────────────────────────────────────────────────────
# This notebook lives in notebooks/ but reads checkpoints/, test_results/ and
# report/ from the project root, and imports data/engine/model from src/.
# Re-running this cell is safe (it only climbs out of notebooks/ once).
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJ = os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

RESULTS_DIR = "test_results"
PRED_FILES  = {"mr0.00": os.path.join(RESULTS_DIR, "best_mr0.00", "predictions.pt"),
               "mr0.50": os.path.join(RESULTS_DIR, "best_mr0.50", "predictions.pt")}
EXCLUDE     = ["PFA"]
MR          = "mr0.50"          # mask ratio used for the main analysis
PRESENT_FRAC_THRESH = 0.50      # ≥ this fraction of training steps present ⇒ "well-trained"
MIN_OBS     = 50                # < this many training obs ⇒ pipeline used the global fallback

from data.dataset import TRAIN_YEARS, VARIABLE_NAMES
VARS  = ["temperature", "pressure", "humidity", "wind_u", "wind_v"]
UNITS = {"temperature":"°C","pressure":"hPa","humidity":"%","wind_u":"m/s","wind_v":"m/s"}

_CANDIDATES = [
    os.environ.get("DATA_ROOT", ""),   # set DATA_ROOT to override"/home/renku/work/PeakWeatherDataset",
               os.path.expanduser("~/Documents/ETH/_DAS Project/PeakWeatherDataset"),
               os.path.expanduser("~/PeakWeatherDataset"), "PeakWeatherDataset"]
DATA_ROOT = next((p for p in _CANDIDATES if os.path.isdir(p)), _CANDIDATES[-1])
plt.rcParams.update({"figure.dpi":110, "font.size":10})
print("Data root:", DATA_ROOT, "(found)" if os.path.isdir(DATA_ROOT) else "(missing)")

In [ ]:
# ── Load predictions + forecast-time labels ──────────────────────────────────
import torch
def load_pred(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    return {k: (v.numpy() if torch.is_tensor(v) else v) for k, v in d.items()}

PRED = {mr: load_pred(p) for mr, p in PRED_FILES.items()}
N_STATIONS = PRED["mr0.00"]["preds"].shape[2]
DELTA_GRID = PRED["mr0.00"]["delta_steps"][0].astype(int)

def fmt_lead(steps):
    m = int(steps) * 10
    if m == 0:      return "t=0"
    if m < 60:      return f"+{m}min"
    if m % 60 == 0: return f"+{m//60}h"
    return f"+{m//60}h{m%60:02d}"
LEAD_LABELS = [fmt_lead(s) for s in DELTA_GRID]
print("Stations:", N_STATIONS, "| horizons:", LEAD_LABELS)

In [ ]:
# ── Training coverage, per-(station,var) physical scale, global reference ─────
from data.dataset import load_peakweather, StationMAEDataset
ds = load_peakweather(root=DATA_ROOT)

order    = ds.stations_table.index.tolist()
keep     = StationMAEDataset._resolve_keep_indices(ds, EXCLUDE)
kept_ids = [order[i] for i in keep]
assert len(kept_ids) == N_STATIONS, f"{len(kept_ids)} kept != {N_STATIONS} predicted"

obs_all  = ds.get_observations(parameters=VARIABLE_NAMES)
obs_all.index = pd.to_datetime(obs_all.index)
train    = obs_all.loc[np.isin(obs_all.index.year, TRAIN_YEARS)]
n_train  = len(train)

def to_mat(series):                       # (station,var) Series → (N,5) aligned array
    return series.unstack().reindex(index=kept_ids, columns=VARS).values

cnt = np.nan_to_num(to_mat(train.notna().sum()), nan=0.0)
mn  = to_mat(train.mean())
sd  = to_mat(train.std())

# Global per-variable physical stats (pooled) — used as fallback + scale reference
g_mean = np.array([np.nanmean(train.xs(v, axis=1, level=1).values) for v in VARS])
g_std  = np.array([np.nanstd (train.xs(v, axis=1, level=1).values) for v in VARS])
g_p01  = np.array([np.nanpercentile(train.xs(v, axis=1, level=1).values, 1)  for v in VARS])
g_p99  = np.array([np.nanpercentile(train.xs(v, axis=1, level=1).values, 99) for v in VARS])

# Exact denorm scale the pipeline used: per-station, global fallback where obs < MIN_OBS
use_global = (cnt < MIN_OBS) | ~np.isfinite(sd) | (sd < 1e-6)
STD_MAT  = np.where(use_global, g_std[None, :],  sd)
MEAN_MAT = np.where(use_global, g_mean[None, :], mn)
STD_MAT  = np.clip(STD_MAT, 1e-6, None)

present_frac     = cnt / n_train
well_trained     = present_frac >= PRESENT_FRAC_THRESH     # (N,5) bool
missing_in_train = ~well_trained
print(f"Well-trained pairs: {int(well_trained.sum())}   |   "
      f"Missing-in-training pairs: {int(missing_in_train.sum())}")

## 1. General metrics per variable — well-trained pairs

Physical-unit MAE / RMSE / bias, restricted to well-trained pairs (`mr0.50`, all lead-times).
The **scale reference** columns — global std σ and the 1–99 % range — plus **nRMSE = RMSE/σ**
(fraction of the variable's natural variability) make the error magnitude interpretable.

In [ ]:
# ── Physical error helper (restricted to a per-variable station subset) ──────
def phys_error(d, vi, subset):
    err  = (d["preds"][..., vi] - d["targets"][..., vi]) * STD_MAT[None, None, :, vi]
    mask = (d["masks"][..., vi] > 0.5) & subset[None, None, :]
    return err, mask

def general_metrics(d, subsets):
    rows = {}
    for vi, v in enumerate(VARS):
        err, mask = phys_error(d, vi, subsets[:, vi])
        e = err[mask]
        if e.size == 0:
            continue
        ae = np.abs(e); rmse = float(np.sqrt(np.mean(e**2)))
        rows[v] = {"MAE": float(ae.mean()), "RMSE": rmse, "bias": float(e.mean()),
                   "σ (global)": float(g_std[vi]),
                   "range 1–99%": f"{g_p01[vi]:.0f} … {g_p99[vi]:.0f}",
                   "nRMSE=RMSE/σ": rmse / float(g_std[vi]),
                   "n_obs": int(e.size)}
    return pd.DataFrame(rows).T

gen = general_metrics(PRED[MR], well_trained)
display(gen.style.format({"MAE":"{:.3f}","RMSE":"{:.3f}","bias":"{:+.3f}",
                          "σ (global)":"{:.2f}","nRMSE=RMSE/σ":"{:.2f}","n_obs":"{:.0f}"})
        .set_caption(f"Well-trained pairs — {MR} (units: "
                     + ", ".join(f"{v} [{UNITS[v]}]" for v in VARS) + ")"))

## 2. Outlier stations / variables

Per `(station, variable)` RMSE (physical) on the well-trained pairs. Within each variable
we flag stations whose RMSE is a statistical **outlier** across the network using the
robust IQR rule: `RMSE > Q3 + 1.5·IQR`. These are the sites the model handles worst.

In [ ]:
# ── Per-(station,variable) RMSE and IQR-based outlier flagging ────────────────
d = PRED[MR]
recs = []
for vi, v in enumerate(VARS):
    err  = (d["preds"][..., vi] - d["targets"][..., vi]) * STD_MAT[None, None, :, vi]  # (M,K,N)
    mk   = (d["masks"][..., vi] > 0.5)
    for ni in range(N_STATIONS):
        if not well_trained[ni, vi]:
            continue
        e = err[:, :, ni][mk[:, :, ni]]
        if e.size < 30:                      # need enough points for a stable RMSE
            continue
        recs.append({"station": kept_ids[ni], "variable": v,
                     "rmse": float(np.sqrt(np.mean(e**2))), "n": int(e.size)})
per_sv = pd.DataFrame(recs)

flagged = []
for v in VARS:
    sub = per_sv[per_sv.variable == v]
    if len(sub) < 4:
        continue
    q1, q3 = np.percentile(sub.rmse, [25, 75]); iqr = q3 - q1
    thr = q3 + 1.5 * iqr
    out = sub[sub.rmse > thr].copy()
    out["var_median_rmse"] = float(sub.rmse.median())
    out["threshold"] = thr
    out["x_median"]   = out.rmse / float(sub.rmse.median())
    flagged.append(out)

flagged = (pd.concat(flagged).sort_values("x_median", ascending=False)
           if flagged else pd.DataFrame())
print(f"Outlier (station, variable) pairs flagged: {len(flagged)}")
if len(flagged):
    display(flagged[["station","variable","rmse","var_median_rmse","x_median","threshold","n"]]
            .head(25).reset_index(drop=True)
            .style.format({"rmse":"{:.3f}","var_median_rmse":"{:.3f}",
                           "x_median":"{:.2f}×","threshold":"{:.3f}","n":"{:.0f}"})
            .set_caption("Stations with outlier RMSE (per variable, physical units)"))

## 3. Average error by forecast timestep and variable

Mean absolute error at each forecast horizon (well-trained pairs). Colour = **nMAE =
MAE/σ** (comparable across variables); the printed number is the physical MAE.

In [ ]:
# ── Average error (MAE) by forecast horizon × variable ───────────────────────
K = len(DELTA_GRID)
mae_mat  = np.full((len(VARS), K), np.nan)
nmae_mat = np.full((len(VARS), K), np.nan)
d = PRED[MR]
for vi, v in enumerate(VARS):
    for k in range(K):
        e = ((d["preds"][:, k, :, vi] - d["targets"][:, k, :, vi]) * STD_MAT[:, vi])
        m = (d["masks"][:, k, :, vi] > 0.5) & well_trained[None, :, vi]
        e = e[m]
        if e.size:
            mae_mat[vi, k]  = np.mean(np.abs(e))
            nmae_mat[vi, k] = mae_mat[vi, k] / g_std[vi]

fig, ax = plt.subplots(figsize=(1.0*K+2, 0.6*len(VARS)+1.5))
im = ax.imshow(nmae_mat, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(K)); ax.set_xticklabels(LEAD_LABELS, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(VARS))); ax.set_yticklabels([f"{v}\n[{UNITS[v]}]" for v in VARS])
for vi in range(len(VARS)):
    for k in range(K):
        if np.isfinite(mae_mat[vi, k]):
            ax.text(k, vi, f"{mae_mat[vi,k]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title(f"Average |error| by forecast horizon — well-trained pairs ({MR})")
plt.colorbar(im, ax=ax, label="nMAE = MAE / σ", shrink=0.8)
plt.tight_layout(); plt.show()

# same values as a table
display(pd.DataFrame(mae_mat, index=VARS, columns=LEAD_LABELS)
        .style.format("{:.3f}").background_gradient(axis=1, cmap="YlOrRd")
        .set_caption("Physical MAE by forecast horizon (well-trained pairs)"))

## 4. Missing-in-training pairs *(kept for reference)*

The same general metrics, now restricted to the `(station, variable)` pairs that are
**missing in training** (present in < 50 % of training years). These used the global
normalisation fallback, so their physical scale is the global σ. Expect weaker, more
biased performance — the model never learned a site-specific representation for them.

In [ ]:
# ── General metrics on the missing-in-training pairs ─────────────────────────
if missing_in_train.any():
    miss = general_metrics(PRED[MR], missing_in_train)
    display(miss.style.format({"MAE":"{:.3f}","RMSE":"{:.3f}","bias":"{:+.3f}",
                               "σ (global)":"{:.2f}","nRMSE=RMSE/σ":"{:.2f}","n_obs":"{:.0f}"})
            .set_caption(f"Missing-in-training pairs — {MR}"))

    # side-by-side nRMSE: well-trained vs missing
    comp = pd.DataFrame({
        "well-trained nRMSE": general_metrics(PRED[MR], well_trained)["nRMSE=RMSE/σ"],
        "missing nRMSE":      miss["nRMSE=RMSE/σ"],
    })
    display(comp.style.format("{:.2f}")
            .set_caption("nRMSE (RMSE/σ): well-trained vs missing-in-training"))
else:
    print("No missing-in-training pairs at the current threshold.")